# MichiganCast Demo 01: Data Foundation

This notebook demonstrates contract driven validation, data quality checks, deterministic cleaning, and data version traceability.

## 0. Objectives and Conclusions

**Objectives**
1. Validate schema and value constraints with explicit failure reporting.
2. Evaluate data quality risks before model training.
3. Run a reproducible cleaning pipeline.
4. Register dataset version metadata for traceability.

**Conclusion Template**
- Which quality issues were detected and whether they block training.
- Which cleaned dataset version is used for downstream tasks.
- Whether artifacts satisfy reproducibility expectations.

## 1. Inputs and Outputs

| Type | Path |
|---|---|
| Contract module | `src/data/contracts.py` |
| Validation module | `src/data/validate.py` |
| Cleaning module | `src/data/clean.py` |
| Versioning module | `src/data/versioning.py` |
| Raw tabular input | `data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv` |
| Clean output | `data/interim/traverse_city_daytime_clean_v1.csv` |
| Reports | `artifacts/reports/*.json` |
| Version manifest | `configs/data/versions/*.json` |

In [ ]:
from pathlib import Path
import subprocess
import json

RAW_CSV = 'data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv'
CLEAN_CSV = 'data/interim/traverse_city_daytime_clean_v1.csv'
IMAGE_DIR = 'data/processed/images/lake_michigan_64_png'

print('RAW exists:', Path(RAW_CSV).exists())
print('IMAGE dir exists:', Path(IMAGE_DIR).exists())
print('CLEAN exists:', Path(CLEAN_CSV).exists())

## 2. Method and Implementation Using src Modules

Execution is disabled by default to keep notebook runs deterministic and fast. Set `DEMO_EXECUTE=True` only when you need to generate fresh artifacts.

In [ ]:
DEMO_EXECUTE = False

def run_or_print(cmd: str):
    print('\n$ ' + cmd)
    if not DEMO_EXECUTE:
        print('[skip] DEMO_EXECUTE=False, only showing command.')
        return None
    completed = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
    return completed.returncode

### 2.1 Data Contract Validation T10

**Purpose** Validate required columns, data types, missing value rules, and physical ranges. The contract report is the first quality gate before any transformation.

In [ ]:
cmd_contract = (
    'scripts/run_in_pytorch_env.sh python -m src.data.contracts '
    '--input data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv '
    '--output artifacts/reports/data_contract_report.json'
)
run_or_print(cmd_contract)

### 2.2 Data Quality Validation T11

**Purpose** Detect missing code patterns, temporal continuity issues, broken images, and tabular image alignment risks.

In [ ]:
cmd_validate = (
    'scripts/run_in_pytorch_env.sh python -m src.data.validate '
    '--input data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv '
    '--image-dir data/processed/images/lake_michigan_64_png '
    '--output artifacts/data_quality_report.json'
)
run_or_print(cmd_validate)

### 2.3 Cleaning Pipeline T12

**Purpose** Convert manual notebook cleaning into a deterministic command line pipeline with explicit summary outputs.

In [ ]:
cmd_clean = (
    'scripts/run_in_pytorch_env.sh python -m src.data.clean '
    '--input data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv '
    '--output data/interim/traverse_city_daytime_clean_v1.csv '
    '--summary artifacts/reports/data_cleaning_summary.json'
)
run_or_print(cmd_clean)

### 2.4 Data Version Tracking T31

**Purpose** Persist dataset lineage so every training run can reference exact input provenance.

In [ ]:
cmd_version = (
    'scripts/run_in_pytorch_env.sh python -m src.data.versioning '
    '--dataset-id traverse_city_clean_v1 '
    '--layer interim '
    '--target-path data/interim/traverse_city_daytime_clean_v1.csv '
    '--source-path data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv '
    '--build-command "python -m src.data.clean --input ... --output ..." '
    '--notes "demo notebook snapshot"'
)
run_or_print(cmd_version)

## 3. Results and Interpretation

Review each generated report and answer three questions:
1. Is the input data contract compliant.
2. Which quality issues remain after cleaning.
3. Is the resulting dataset version ready for modeling.

In [ ]:
candidate_reports = [
    'artifacts/reports/data_contract_report.json',
    'artifacts/data_quality_report.json',
    'artifacts/reports/data_cleaning_summary.json',
    'configs/data/versions/traverse_city_clean_v1.json',
]

for rp in candidate_reports:
    p = Path(rp)
    if not p.exists():
        print(f'{rp}: NOT FOUND')
        continue
    print(f'{rp}: FOUND, size={p.stat().st_size} bytes')

## 4. Engineering Notes and Next Steps

- Promote blocking checks to non zero exit status when strict mode is enabled.
- Add schema drift alerts between dataset versions.
- Integrate report validation into CI smoke tests.

## Appendix. Reproducible Commands

```bash
scripts/run_in_pytorch_env.sh python -m src.data.contracts --input data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv --output artifacts/reports/data_contract_report.json
scripts/run_in_pytorch_env.sh python -m src.data.validate --input data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv --image-dir data/processed/images/lake_michigan_64_png --output artifacts/data_quality_report.json
scripts/run_in_pytorch_env.sh python -m src.data.clean --input data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv --output data/interim/traverse_city_daytime_clean_v1.csv --summary artifacts/reports/data_cleaning_summary.json
scripts/run_in_pytorch_env.sh python -m src.data.versioning --input data/interim/traverse_city_daytime_clean_v1.csv --output configs/data/versions/traverse_city_daytime_clean_v1.json
```